<a href="https://colab.research.google.com/github/divya123amar/APMC-react-JS/blob/main/monai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ===============================
# 1. INSTALL LIBRARIES
# ===============================
!pip install -q monai nibabel torch torchvision tqdm matplotlib

In [10]:
# ===============================
# 2. IMPORT LIBRARIES
# ===============================

import os
import glob
import tarfile
import urllib.request
import numpy as np
import torch
import matplotlib.pyplot as plt
import nibabel as nib

from tqdm import tqdm

from monai.utils import set_determinism

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    EnsureTyped,
    AsDiscrete
)

from monai.data import Dataset, DataLoader, decollate_batch

from monai.networks.nets import UNet

from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference

In [11]:
# ===============================
# 3. DEVICE SETUP
# ===============================

set_determinism(seed=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [12]:
# ===============================
# 4. CREATE OUTPUT DIRECTORIES
# ===============================

BASE_DIR = "edge_output"

MODEL_DIR = os.path.join(BASE_DIR, "models")
IMAGE_DIR = os.path.join(BASE_DIR, "predictions")
GRAPH_DIR = os.path.join(BASE_DIR, "graphs")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(GRAPH_DIR, exist_ok=True)

print("Folders created.")

Folders created.


In [13]:
# ===============================
# 5. DOWNLOAD DATASET
# ===============================

url = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar"
filename = "Task01_BrainTumour.tar"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

if not os.path.exists("Task01_BrainTumour"):
    tar = tarfile.open(filename)
    tar.extractall()
    tar.close()

data_dir = "./Task01_BrainTumour"

print("Dataset ready")

Dataset ready


In [14]:
# ===============================
# 6. DATASET PREPARATION
# ===============================

images = sorted(glob.glob(os.path.join(data_dir, "imagesTr", "*.nii.gz")))
labels = sorted(glob.glob(os.path.join(data_dir, "labelsTr", "*.nii.gz")))

data_dicts = [
    {"image": img, "label": lbl}
    for img, lbl in zip(images, labels)
]

train_files = data_dicts[:80]
val_files = data_dicts[80:100]

print("Train:", len(train_files))
print("Validation:", len(val_files))

Train: 0
Validation: 0


In [15]:
# ===============================
# 7. TRANSFORMS (FIXED)
# ===============================

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    EnsureTyped,
    ResizeWithPadOrCropd,
    ScaleIntensityRanged
)

train_transforms = Compose([
    LoadImaged(keys=["image","label"]),

    EnsureChannelFirstd(keys=["image","label"]),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=0,
        a_max=3000,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),

    # IMPORTANT FIX
    ResizeWithPadOrCropd(
        keys=["image","label"],
        spatial_size=(128,128,128)
    ),

    EnsureTyped(keys=["image","label"])
])


val_transforms = Compose([
    LoadImaged(keys=["image","label"]),

    EnsureChannelFirstd(keys=["image","label"]),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=0,
        a_max=3000,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),

    ResizeWithPadOrCropd(
        keys=["image","label"],
        spatial_size=(128,128,128)
    ),

    EnsureTyped(keys=["image","label"])
])

In [17]:
# ===============================
# 8. DATA LOADERS
# ===============================

train_ds = Dataset(train_files, train_transforms)
val_ds = Dataset(val_files, val_transforms)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1)

print("Dataset ready")

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# ===============================
# 9. MODEL
# ===============================

model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16,32,64,128,256),
    strides=(2,2,2,2)
).to(device)

In [ ]:
# ===============================
# 10. LOSS + OPTIMIZER
# ===============================

loss_function = DiceLoss(to_onehot_y=True, softmax=True)

optimizer = torch.optim.Adam(model.parameters(), 1e-4)

In [ ]:
# ===============================
# 11. METRICS
# ===============================

dice_metric = DiceMetric(include_background=True, reduction="mean")

post_pred = Compose([AsDiscrete(argmax=True, to_onehot=4)])

post_label = Compose([AsDiscrete(to_onehot=4)])

In [ ]:
# ===============================
# 12. TRAIN FUNCTION
# ===============================

def train_epoch(model, loader, optimizer, loss_function):

    model.train()

    epoch_loss = 0

    for batch_data in loader:

        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = loss_function(outputs, labels)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss = epoch_loss / len(loader)

    return epoch_loss

In [ ]:
# ===============================
# 13. VALIDATION FUNCTION
# ===============================

def validate_epoch(model, loader):

    model.eval()

    dice_metric.reset()

    with torch.no_grad():

        for val_data in loader:

            val_inputs = val_data["image"].to(device)
            val_labels = val_data["label"].to(device)

            val_outputs = sliding_window_inference(
                val_inputs,
                (96,96,96),
                1,
                model
            )

            val_outputs = [post_pred(i) for i in decollate_batch(val_outputs)]
            val_labels = [post_label(i) for i in decollate_batch(val_labels)]

            dice_metric(y_pred=val_outputs, y=val_labels)

    metric = dice_metric.aggregate().item()

    dice_metric.reset()

    return metric

In [ ]:
# ===============================
# 14. TRAINING LOOP
# ===============================

max_epochs = 10

best_metric = -1

train_losses = []
val_dice_scores = []

for epoch in range(max_epochs):

    print("Epoch:", epoch+1)

    train_loss = train_epoch(model, train_loader, optimizer, loss_function)

    val_metric = validate_epoch(model, val_loader)

    train_losses.append(train_loss)
    val_dice_scores.append(val_metric)

    print("Train Loss:", train_loss)
    print("Validation Dice:", val_metric)

    if val_metric > best_metric:

        best_metric = val_metric

        torch.save(model.state_dict(), os.path.join(MODEL_DIR,"best_model.pth"))

        print("Best Model Saved")

In [ ]:
# ===============================
# 15. SAVE MODEL
# ===============================

torch.save(model.state_dict(), os.path.join(MODEL_DIR,"final_model.pth"))

print("Model saved")

In [ ]:
# ===============================
# 16. QUANTIZE MODEL
# ===============================

quantized_model = torch.quantization.quantize_dynamic(
    model.cpu(),
    {torch.nn.Linear},
    dtype=torch.qint8
)

torch.save(
    quantized_model.state_dict(),
    os.path.join(MODEL_DIR,"quantized_model.pth")
)

print("Quantized model saved")

In [ ]:
# ===============================
# 17. SAVE PREDICTIONS
# ===============================

model.eval()

with torch.no_grad():

    for i, val_data in enumerate(val_loader):

        val_inputs = val_data["image"].to(device)

        val_outputs = sliding_window_inference(
            val_inputs,
            (96,96,96),
            1,
            model
        )

        pred = torch.argmax(val_outputs, dim=1)

        pred_np = pred.cpu().numpy()[0]

        save_path = os.path.join(IMAGE_DIR, f"prediction_{i}.nii.gz")

        nib.save(nib.Nifti1Image(pred_np.astype(np.float32), np.eye(4)), save_path)

print("Predictions saved")

In [ ]:
# ===============================
# 18. TRAINING GRAPHS
# ===============================

plt.figure()

plt.plot(train_losses)

plt.title("Training Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.savefig(os.path.join(GRAPH_DIR,"training_loss.png"))

plt.show()


plt.figure()

plt.plot(val_dice_scores)

plt.title("Validation Dice")

plt.xlabel("Epoch")

plt.ylabel("Dice")

plt.savefig(os.path.join(GRAPH_DIR,"validation_dice.png"))

plt.show()

print("Graphs saved")

In [ ]:
# ===============================
# 17. SAVE PREDICTIONS AS JPG
# ===============================

import cv2
import os

model.eval()

with torch.no_grad():

    for i, val_data in enumerate(val_loader):

        val_inputs = val_data["image"].to(device)

        val_outputs = sliding_window_inference(
            val_inputs,
            (96,96,96),
            1,
            model
        )

        pred = torch.argmax(val_outputs, dim=1)

        pred_np = pred.cpu().numpy()[0]   # shape: (H, W, D)

        # create folder for each volume
        volume_dir = os.path.join(IMAGE_DIR, f"prediction_{i}")
        os.makedirs(volume_dir, exist_ok=True)

        depth = pred_np.shape[2]

        for slice_id in range(depth):

            slice_img = pred_np[:, :, slice_id]

            # normalize to 0–255
            slice_img = (slice_img / slice_img.max()) * 255
            slice_img = slice_img.astype("uint8")

            save_path = os.path.join(volume_dir, f"slice_{slice_id}.jpg")

            cv2.imwrite(save_path, slice_img)

print("Predictions saved as JPG slices.")